# MP Surya-Drishti — Native-Resolution Tiling & Focal-Dice Training
## Production SegFormer-B2 Training on Google Colab (GPU)

This notebook orchestrates the complete end-to-end training and evaluation pipeline for **MP Surya-Drishti**:
1. **Environment Setup**: GPU verification & dependency installation.
2. **Dataset Audit**: Official Massachusetts split pairing & data-driven class imbalance measurement.
3. **Native-Resolution Training (`exp_003`)**: SegFormer-B2 trained on native $512 \times 512$ patches with **Focal-Dice loss** ($50\%$ tile overlap, 50 epochs).
4. **Sliding-Window Evaluation**: Evaluation using Gaussian blending at $1500 \times 1500$ native resolution.
5. **Benchmark Comparison**: Scientific matrix comparing `exp_002` (old baseline) vs `exp_003` (native-resolution upgrade).

### 1. Verify Colab GPU Acceleration

In [ ]:
# Check CUDA GPU availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Mount Google Drive & Set Workspace Directory

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Update path below to your exact Google Drive project directory
PROJECT_DIR = '/content/drive/MyDrive/MP_Surya_Drishti/rooftop_segmentation'
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f"Active working directory: {os.getcwd()}")
else:
    print(f"Warning: {PROJECT_DIR} not found. Please set correct path or clone repo.")

### 3. Install Framework Dependencies

In [ ]:
!pip install -r requirements.txt --quiet
!pip install albumentations>=1.3.1 transformers>=4.36.0 torchvision>=0.16.0 --quiet

### 4. Verify Dataset Splits & Measure Class Distribution

In [ ]:
# 1. Verify 137 Train / 4 Val / 10 Test image-mask pairing integrity
!python main.py verify-dataset

# 2. Measure exact ground truth pixel imbalance across training set
!python main.py measure-imbalance

### 5. Launch Native-Resolution Tiled Training (`exp_003`)
Trains SegFormer-B2 on native $512 \times 512$ crops with **Focal-Dice loss** into `outputs/experiments/exp_003/`.

In [ ]:
!python main.py train \
    --tiled \
    --tile-size 512 \
    --stride 256 \
    --loss focal_dice \
    --epochs 50 \
    --batch-size 8 \
    --exp-name exp_003

### 6. Evaluate `exp_003` Model with Sliding-Window Inference

In [ ]:
# Evaluate native-resolution tiled inference (Gaussian blending)
!python main.py evaluate \
    --checkpoint outputs/experiments/exp_003/checkpoints/best_loss.pth \
    --tiled \
    --tile-size 512 \
    --stride 256 \
    --blend-mode gaussian \
    --output-dir outputs/reports/exp_003_evaluation

### 7. Generate Multi-Mode Scientific Showcase & Benchmark Matrix
Generates comparative charts, 5-panel debug figures, and comparison table against `exp_002` baseline.

In [ ]:
!python test_evaluation_showcase.py \
    --checkpoint outputs/experiments/exp_003/checkpoints/best_loss.pth \
    --output-dir outputs \
    --stride 256 \
    --tta

### 8. Display Visual Benchmark Charts Inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path

chart_paths = [
    "outputs/charts/pipeline_comparison_chart.png",
    "outputs/charts/tile_grid_visualization.png",
    "outputs/charts/metrics_bar_chart.png",
    "outputs/charts/per_image_iou_chart.png",
    "outputs/charts/confusion_matrix.png",
]

for cp in chart_paths:
    if Path(cp).exists():
        print(f"\n=== {Path(cp).name} ===")
        display(Image(filename=cp))
    else:
        print(f"Chart not found: {cp}")